# LSC5 | Preentrenamiento con ISLR (datos externos) y ajuste fino con LSC5

**EXPERIMENTO SEPARADO DEL NOTEBOOK OFICIAL.** Usa datos externos (Google *Isolated Sign Language Recognition*, ASL, Kaggle, CC-BY 4.0 segun la tarjeta del dataset).
Las reglas del reto piden **declarar** los datos externos y quedan a criterio del comite. Si no se aprueban, este notebook no se usa para la entrega.

Que hace:
1. Convierte secuencias de ISLR (parquet, 543 puntos) al mismo formato (T, 1662) de LSC5 y a las mismas 400 caracteristicas.
2. Preentrena el TCN para clasificar las 250 senas de ISLR (con espejo).
3. Ajusta fino con LSC5 en LOSO (M02/M03/M04) y lo compara **con el mismo modelo entrenado desde cero**, misma configuracion y semillas.

Escalador: se ajusta **solo con ISLR** y se congela para todo (ni LSC5 ni el test intervienen), asi no hay fuga alguna.
Regla de integridad: el test de LSC5 no se lee aqui.

Necesita en Kaggle: la competicion `asl-signs` (Add Input -> Competition; hay que unirse y aceptar sus reglas) y el dataset privado con los landmarks de LSC5.

In [ ]:
from pathlib import Path
import os, re, json, math, random, time, copy, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from torch import nn
from IPython.display import display
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------
# Configuración editable
# -----------------------------
DATA_ROOT = Path('/kaggle/input')
TRAIN_LANDMARKS_DIR = None  # Ejemplo: Path('/kaggle/input/lsc5/train/landmarks')
TEST_LANDMARKS_DIR = None   # Se usa únicamente en la celda opcional de inferencia final.

SEED = 42
SEQ_LEN = 64
BATCH_SIZE = 32
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
TCN_CHANNELS = 96
DROPOUT = 0.25
NUM_WORKERS = 2
USE_AMP = True

# Lateralidad: M02 es zurdo y M03/M04 son diestros. Espejar (aumento + promedio al predecir)
# subio la accuracy LOSO media de ~0.72 a ~0.91 y la de M02 de ~0.45 a ~0.95.
MIRROR_AUG = True   # entrenar con cada muestra y su version espejada
MIRROR_TTA = True   # al predecir, promediar probabilidades de la muestra y su espejo

# Debe permanecer False durante desarrollo/validación.
RUN_FINAL_TEST_INFERENCE = False

OUT_DIR = Path('/kaggle/working/lsc5_islr_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [f'sign_{i:02d}' for i in range(1, 6)]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def seed_everything(seed=SEED):
    """Fija semillas para hacer los experimentos lo más reproducibles posible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Se prioriza reproducibilidad; puede ser algo más lento que el modo benchmark.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print('PyTorch:', torch.__version__)
print('Dispositivo:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No se detectó GPU. Para el TCN, activa GPU en Kaggle si está disponible.')
print('Directorio de artefactos:', OUT_DIR)

# ---- Parametros del experimento ----
ISLR_DIR = None            # p. ej. Path('/kaggle/input/asl-signs'); None = autodeteccion
ISLR_MAX_SEQ = 24000       # secuencias de ISLR a usar (muestreo balanceado por sena)
PRETRAIN_EPOCHS = 15
PRETRAIN_LR = 1e-3
PRETRAIN_BATCH = 256
FT_EPOCHS = 12             # epocas de ajuste fino en cada fold LOSO
FT_LR = 5e-4
SEEDS = [42, 7, 123]


In [ ]:
def _class_dirs(path: Path):
    if not path.exists() or not path.is_dir():
        return []
    return [p for p in path.iterdir() if p.is_dir() and re.fullmatch(r'sign_\d{2}', p.name)]


def discover_train_landmarks(root=DATA_ROOT, override=None):
    """Encuentra train/landmarks sin entrar en carpetas de test."""
    if override is not None:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f'La ruta TRAIN_LANDMARKS_DIR no existe: {p}')
        return p

    if not root.exists():
        raise FileNotFoundError(f'No existe DATA_ROOT: {root}')

    candidates = []
    for p in root.rglob('landmarks'):
        if not p.is_dir():
            continue
        parts_lower = [part.lower() for part in p.parts]
        if 'test' in parts_lower:
            continue
        classes = _class_dirs(p)
        if len(classes) >= 5 and any(p.rglob('*.npy')):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            'No encontré una carpeta train/landmarks con subcarpetas sign_01 ... sign_05. '
            'Define TRAIN_LANDMARKS_DIR manualmente.'
        )

    # Preferir una ruta que contenga explícitamente "train".
    candidates.sort(key=lambda p: ('train' not in [x.lower() for x in p.parts], len(str(p))))
    return candidates[0]


def parse_sample_metadata(path: Path):
    """Extrae etiqueta, participante, vista, cámara y grupo de ejecución."""
    stem = path.stem
    parent_match = re.fullmatch(r'(sign_\d{2})', path.parent.name)
    sign_match = re.search(r'(?:^|_)S(\d{2})(?:_|$)', stem)
    label = parent_match.group(1) if parent_match else (f'sign_{int(sign_match.group(1)):02d}' if sign_match else None)

    signer_match = re.search(r'(?:^|_)M(\d{2})(?:_|$)', stem)
    if signer_match is None:
        raise ValueError(f'No pude extraer participante M## del nombre: {path.name}')
    signer = f'M{signer_match.group(1)}'

    view_match = re.search(r'(?:^|_)V(\d{2})(?:_|$)', stem)
    camera_match = re.search(r'(?:^|_)C(\d{2})(?:_|$)', stem)
    clip_match = re.search(r'Clip(\d+)', stem, flags=re.IGNORECASE)

    if label not in CLASS_NAMES:
        raise ValueError(f'No pude identificar una clase sign_01 ... sign_05: {path}')

    # Agrupa las vistas/cámaras que comparten el mismo clip. Si no hay Clip, usa el archivo.
    group_id = f'{label}_{signer}_Clip{clip_match.group(1)}' if clip_match else stem
    return {
        'path': str(path),
        'video_id': stem,
        'label': label,
        'y': CLASS_TO_ID[label],
        'signer': signer,
        'view': f'V{view_match.group(1)}' if view_match else 'unknown',
        'camera': f'C{camera_match.group(1)}' if camera_match else 'unknown',
        'clip': clip_match.group(1) if clip_match else None,
        'group_id': group_id,
    }




In [ ]:
# Índices de la representación descrita en el manual LSC5 / MediaPipe Holistic.
POSE_SLICE = slice(0, 132)
FACE_SLICE = slice(132, 1536)
LEFT_HAND_SLICE = slice(1536, 1599)
RIGHT_HAND_SLICE = slice(1599, 1662)

BODY_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24]  # hombros, codos, muñecas y caderas
FACE_JOINTS = [1, 33, 133, 362, 263, 61, 291, 13, 14, 70, 300]
BASE_FEATURE_DIM = 200
FEATURE_DIM = BASE_FEATURE_DIM * 2  # posición + velocidad
MASK_FEATURES = [197, 198, 199, 397, 398, 399]


def load_landmark_array(path):
    """Carga un .npy y valida la forma esperada (T, 1662)."""
    arr = np.load(path, allow_pickle=False)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'{path}: esperaba matriz 2D, recibí {arr.shape}')
    if arr.shape[1] != 1662 and arr.shape[0] == 1662:
        arr = arr.T
    if arr.shape[1] != 1662:
        raise ValueError(f'{path}: esperaba 1662 características por frame, recibí {arr.shape}')
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if len(arr) < 2:
        raise ValueError(f'{path}: secuencia demasiado corta ({len(arr)} frames)')
    return arr


def resample_frames(arr, target_len=SEQ_LEN):
    """Remuestreo lineal vectorizado a una longitud temporal fija."""
    n = len(arr)
    if n == target_len:
        return arr.astype(np.float32, copy=False)
    positions = np.linspace(0, n - 1, target_len, dtype=np.float32)
    lo = np.floor(positions).astype(np.int64)
    hi = np.minimum(lo + 1, n - 1)
    alpha = (positions - lo).astype(np.float32)[:, None]
    return (arr[lo] * (1.0 - alpha) + arr[hi] * alpha).astype(np.float32)


def landmarks_to_features(arr, target_len=SEQ_LEN):
    """Convierte landmarks crudos en una secuencia (target_len, 400)."""
    arr = resample_frames(arr, target_len)
    t = len(arr)

    pose = arr[:, POSE_SLICE].reshape(t, 33, 4)
    face = arr[:, FACE_SLICE].reshape(t, 468, 3)
    left_hand = arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3)
    right_hand = arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3)

    # Centro del torso y escala robusta por video: distancia entre hombros.
    left_shoulder = pose[:, 11, :3]
    right_shoulder = pose[:, 12, :3]
    torso_center = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder[:, :2] - right_shoulder[:, :2], axis=1)
    valid_widths = shoulder_width[shoulder_width > 1e-4]
    body_scale = float(np.median(valid_widths)) if len(valid_widths) else 1.0
    body_scale = max(body_scale, 1e-3)

    def hand_features(hand):
        present = (np.max(np.abs(hand), axis=(1, 2)) > 1e-6).astype(np.float32)
        wrist = hand[:, 0, :]
        local = (hand - wrist[:, None, :]) / body_scale
        global_wrist = (wrist - torso_center) / body_scale
        local[present == 0] = 0.0
        global_wrist[present == 0] = 0.0
        return local.reshape(t, -1), global_wrist, present

    left_local, left_wrist, left_present = hand_features(left_hand)
    right_local, right_wrist, right_present = hand_features(right_hand)

    # Postura relativa al torso; se conserva visibilidad como cuarta característica.
    body_xyz = (pose[:, BODY_JOINTS, :3] - torso_center[:, None, :]) / body_scale
    body_visibility = pose[:, BODY_JOINTS, 3:4]
    body_features = np.concatenate([body_xyz, body_visibility], axis=2).reshape(t, -1)

    # Rasgos faciales compactos. Se conservan cambios locales respecto a la nariz.
    selected_face = face[:, FACE_JOINTS, :]
    nose = face[:, 1, :]
    eye_width = np.linalg.norm(face[:, 33, :2] - face[:, 263, :2], axis=1)
    valid_eye_widths = eye_width[eye_width > 1e-4]
    face_scale = float(np.median(valid_eye_widths)) if len(valid_eye_widths) else 1.0
    face_scale = max(face_scale, 1e-3)
    face_present = (np.max(np.abs(face), axis=(1, 2)) > 1e-6).astype(np.float32)
    face_local = (selected_face - nose[:, None, :]) / face_scale
    face_local[face_present == 0] = 0.0
    face_features = face_local.reshape(t, -1)

    # 126 mano + 6 trayectorias de muñeca + 32 cuerpo + 33 rostro + 3 máscaras = 200.
    base = np.concatenate([
        left_local, right_local,
        left_wrist, right_wrist,
        body_features, face_features,
        left_present[:, None], right_present[:, None], face_present[:, None]
    ], axis=1).astype(np.float32)

    if base.shape[1] != BASE_FEATURE_DIM:
        raise RuntimeError(f'Dimensión base inesperada: {base.shape[1]}')

    # Diferencias frame a frame. Las máscaras no representan velocidad.
    velocity = np.diff(base, axis=0, prepend=base[:1]).astype(np.float32)
    velocity[:, -3:] = 0.0
    features = np.concatenate([base, velocity], axis=1).astype(np.float32)
    if features.shape != (target_len, FEATURE_DIM):
        raise RuntimeError(f'Dimensión final inesperada: {features.shape}')
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)




In [ ]:
# 3b. Espejo (lateralidad). Transformacion determinista aplicada a cada muestra por separado.
PAIRS = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),(17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
FACE_PAIRS = [(33,263),(133,362),(61,291),(70,300)]  # solo los puntos de rostro que usa el modelo


def _flipx(block):
    """x -> 1-x solo donde el punto existe (los ceros significan 'no detectado')."""
    block = block.copy()
    present = np.abs(block).max(axis=2) > 1e-6
    block[..., 0] = np.where(present, 1.0 - block[..., 0], 0.0)
    return block


def mirror_raw(arr):
    t = len(arr)
    pose = _flipx(arr[:, POSE_SLICE].reshape(t, 33, 4))
    face = _flipx(arr[:, FACE_SLICE].reshape(t, 468, 3))
    lh = _flipx(arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3))
    rh = _flipx(arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3))
    for i, j in PAIRS:
        pose[:, [i, j]] = pose[:, [j, i]]
    for i, j in FACE_PAIRS:
        face[:, [i, j]] = face[:, [j, i]]
    # al espejar, la mano izquierda pasa a ser la derecha y viceversa
    return np.concatenate([pose.reshape(t, -1), face.reshape(t, -1),
                           rh.reshape(t, -1), lh.reshape(t, -1)], axis=1).astype(np.float32)




In [ ]:
def fit_sequence_scaler(X_train):
    """Ajusta media y desviación por característica usando solo X_train."""
    flat = X_train.reshape(-1, X_train.shape[-1]).astype(np.float64)
    mean = flat.mean(axis=0).astype(np.float32)
    std = flat.std(axis=0).astype(np.float32)
    std[std < 1e-5] = 1.0
    return {'mean': mean, 'std': std}


def apply_sequence_scaler(X_data, scaler):
    return ((X_data - scaler['mean'][None, None, :]) /
            scaler['std'][None, None, :]).astype(np.float32)


def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }


def print_confusion(y_true, y_pred, title='Matriz de confusión'):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
    ax.set_xticks(range(5), CLASS_NAMES, rotation=35, ha='right')
    ax.set_yticks(range(5), CLASS_NAMES)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()
    return cm



In [ ]:
class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.25):
        super().__init__()
        padding = dilation
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.Dropout(dropout),
        )
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(x + self.block(x))


# Los índices corresponden al orden creado en landmarks_to_features:
# manos (0:132), cuerpo (132:164), rostro (164:197), máscaras (197:200),
# seguidos por sus velocidades (200:400).
HAND_STREAM_IDX = list(range(0, 132)) + list(range(200, 332)) + [197, 198]
BODY_STREAM_IDX = list(range(132, 164)) + list(range(332, 364))
FACE_STREAM_IDX = list(range(164, 197)) + list(range(364, 397)) + [199]


class LSC5_TCN(nn.Module):
    """TCN multistream: codificadores separados y fusión temporal posterior."""
    def __init__(self, input_dim=FEATURE_DIM, channels=TCN_CHANNELS,
                 num_classes=5, dropout=DROPOUT):
        super().__init__()
        if input_dim != FEATURE_DIM:
            raise ValueError(f'input_dim debe ser {FEATURE_DIM}, recibí {input_dim}')
        if channels % 3 != 0:
            raise ValueError('TCN_CHANNELS debe ser múltiplo de 3 para las tres ramas.')

        self.register_buffer('hand_idx', torch.tensor(HAND_STREAM_IDX, dtype=torch.long))
        self.register_buffer('body_idx', torch.tensor(BODY_STREAM_IDX, dtype=torch.long))
        self.register_buffer('face_idx', torch.tensor(FACE_STREAM_IDX, dtype=torch.long))
        branch_channels = channels // 3

        def stream_encoder(input_features):
            return nn.Sequential(
                nn.Conv1d(input_features, branch_channels, kernel_size=1),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
            )

        self.hand_encoder = stream_encoder(len(HAND_STREAM_IDX))
        self.body_encoder = stream_encoder(len(BODY_STREAM_IDX))
        self.face_encoder = stream_encoder(len(FACE_STREAM_IDX))

        self.temporal = nn.Sequential(*[
            TemporalResidualBlock(channels, dilation=d, dropout=dropout)
            for d in [1, 2, 4, 8]
        ])
        self.attention = nn.Conv1d(channels, 1, kernel_size=1)
        self.classifier = nn.Sequential(
            nn.Linear(channels * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (batch, tiempo, características). Cada rama recibe su propio subconjunto.
        hand = x.index_select(2, self.hand_idx).transpose(1, 2)
        body = x.index_select(2, self.body_idx).transpose(1, 2)
        face = x.index_select(2, self.face_idx).transpose(1, 2)

        # Codificadores anatómicos independientes.
        streams = torch.cat([
            self.hand_encoder(hand),
            self.body_encoder(body),
            self.face_encoder(face),
        ], dim=1)

        # Fusión de ramas y aprendizaje de dependencias temporales multiescala.
        features = self.temporal(streams)
        weights = torch.softmax(self.attention(features), dim=-1)
        attended = torch.sum(features * weights, dim=-1)
        maximum = torch.max(features, dim=-1).values
        pooled = torch.cat([attended, maximum], dim=1)
        return self.classifier(pooled)




In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X_data, y_data, augment=False):
        self.X = torch.from_numpy(np.asarray(X_data, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y_data, dtype=np.int64))
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def _augment_sequence(self, x):
        # Deformación temporal monotónica suave.
        if random.random() < 0.65:
            t = x.shape[0]
            gamma = random.uniform(0.90, 1.10)
            positions = (np.linspace(0, 1, t, dtype=np.float32) ** gamma) * (t - 1)
            lo = np.floor(positions).astype(np.int64)
            hi = np.minimum(lo + 1, t - 1)
            alpha = (positions - lo)[:, None]
            x = x[lo] * (1.0 - alpha) + x[hi] * alpha

        # Ruido pequeño solo en variables continuas, no en máscaras binarias.
        if random.random() < 0.70:
            noise = np.random.normal(0, 0.012, size=x.shape).astype(np.float32)
            noise[:, MASK_FEATURES] = 0.0
            x = x + noise
        return x.astype(np.float32)

    def __getitem__(self, idx):
        x = self.X[idx].numpy().copy()
        if self.augment:
            x = self._augment_sequence(x)
        return torch.from_numpy(x), self.y[idx]


def make_loader(X_data, y_data, train=False, batch_size=None):
    dataset = SequenceDataset(X_data, y_data, augment=train)
    return DataLoader(
        dataset,
        batch_size=batch_size or BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        drop_last=False,
    )


def predict_tcn(model, X_scaled, batch_size=64):
    model.eval()
    outputs = []
    dummy_y = np.zeros(len(X_scaled), dtype=np.int64)
    loader = DataLoader(SequenceDataset(X_scaled, dummy_y, augment=False),
                        batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            outputs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(outputs, axis=0) if outputs else np.empty((0, 5), dtype=np.float32)


def predict_proba(model, scaler, X_raw, X_mir=None):
    """Probabilidades con escalador congelado; promedia con el espejo si MIRROR_TTA."""
    p = predict_tcn(model, apply_sequence_scaler(X_raw, scaler))
    if MIRROR_TTA and X_mir is not None:
        p = (p + predict_tcn(model, apply_sequence_scaler(X_mir, scaler))) / 2.0
    return p




## 1. Cargar ISLR y convertirlo al formato LSC5

Cada parquet de ISLR tiene filas `frame, type, landmark_index, x, y, z` (543 por fotograma). Se reordena a la disposicion de LSC5: pose (33x4, visibilidad = 1 donde existe), rostro (468x3), mano izquierda (21x3) y mano derecha (21x3). Los NaN (no detectado) pasan a 0, igual que en LSC5.

In [ ]:
import pyarrow.parquet as pq

ISLR_ORDER = {'face': 0, 'left_hand': 1, 'pose': 2, 'right_hand': 3}
ISLR_SIZES = {'face': 468, 'left_hand': 21, 'pose': 33, 'right_hand': 21}


def find_islr_dir(root=DATA_ROOT, override=None):
    if override is not None:
        return Path(override)
    for p in root.rglob('train.csv'):
        if (p.parent / 'sign_to_prediction_index_map.json').exists() or (p.parent / 'train_landmark_files').exists():
            return p.parent
    raise FileNotFoundError('No encontre ISLR (train.csv). Agrega la competicion asl-signs o define ISLR_DIR.')


def islr_parquet_to_lsc5(path):
    """parquet de ISLR -> matriz (T, 1662) con el diseno de LSC5."""
    df = pq.read_table(path, columns=['frame', 'type', 'landmark_index', 'x', 'y', 'z']).to_pandas()
    df['t'] = df['type'].map(ISLR_ORDER)
    df = df.sort_values(['frame', 't', 'landmark_index'])
    frames = df['frame'].nunique()
    if len(df) != frames * 543:
        raise ValueError(f'{path}: {len(df)} filas para {frames} fotogramas (esperaba 543 por fotograma)')
    arr = df[['x', 'y', 'z']].to_numpy(dtype=np.float32).reshape(frames, 543, 3)
    present = np.isfinite(arr).all(axis=2)
    arr = np.nan_to_num(arr, nan=0.0)
    face, lh, pose, rh = arr[:, :468], arr[:, 468:489], arr[:, 489:522], arr[:, 522:]
    vis = present[:, 489:522].astype(np.float32)[..., None]
    pose4 = np.concatenate([pose, vis], axis=2)
    return np.concatenate([pose4.reshape(frames, -1), face.reshape(frames, -1),
                           lh.reshape(frames, -1), rh.reshape(frames, -1)], axis=1).astype(np.float32)


ISLR_DIR = find_islr_dir(DATA_ROOT, ISLR_DIR)
islr_meta = pd.read_csv(ISLR_DIR / 'train.csv')
print('ISLR:', ISLR_DIR, '| secuencias:', len(islr_meta), '| senas:', islr_meta['sign'].nunique(),
      '| firmantes:', islr_meta['participant_id'].nunique())

# Muestreo balanceado por sena (mismo numero maximo por sena), semilla fija.
per_sign = max(1, ISLR_MAX_SEQ // islr_meta['sign'].nunique())
islr_sel = (islr_meta.sample(frac=1, random_state=SEED)
            .groupby('sign').head(per_sign).reset_index(drop=True))
sign_to_id = {s: i for i, s in enumerate(sorted(islr_sel['sign'].unique()))}
print('Secuencias ISLR usadas:', len(islr_sel), '| clases:', len(sign_to_id))

xs, xms, ys = [], [], []
for row in tqdm(islr_sel.itertuples(index=False), total=len(islr_sel), desc='ISLR -> LSC5'):
    try:
        raw = islr_parquet_to_lsc5(ISLR_DIR / row.path)
        if len(raw) < 2:
            continue
        xs.append(landmarks_to_features(raw, SEQ_LEN))
        xms.append(landmarks_to_features(mirror_raw(raw), SEQ_LEN))
        ys.append(sign_to_id[row.sign])
    except Exception as e:
        warnings.warn(f'{row.path}: {e}')
X_islr = np.stack(xs).astype(np.float32); Xm_islr = np.stack(xms).astype(np.float32)
y_islr = np.array(ys, dtype=np.int64)
del xs, xms
print('X_islr:', X_islr.shape, '| y_islr:', y_islr.shape)


## 2. Cargar LSC5 (solo el conjunto etiquetado de entrenamiento)

In [ ]:
TRAIN_LANDMARKS_DIR = discover_train_landmarks(DATA_ROOT, TRAIN_LANDMARKS_DIR)
metadata = pd.DataFrame([parse_sample_metadata(p) for p in sorted(TRAIN_LANDMARKS_DIR.rglob('*.npy'))])
print('LSC5:', len(metadata), 'muestras | participantes', sorted(metadata.signer.unique()))
assert metadata['signer'].nunique() <= 3, 'train no debe incluir al participante de test'
X_l = np.stack([landmarks_to_features(load_landmark_array(p), SEQ_LEN) for p in tqdm(metadata['path'])]).astype(np.float32)
Xm_l = np.stack([landmarks_to_features(mirror_raw(load_landmark_array(p)), SEQ_LEN) for p in tqdm(metadata['path'])]).astype(np.float32)
y_l = metadata['y'].to_numpy(dtype=np.int64)
signers = metadata['signer'].to_numpy()
print('X_l:', X_l.shape)


## 3. Escalador congelado (solo ISLR) y entrenamiento generico

In [ ]:
# Escalador ajustado unicamente con datos externos (ISLR); se reutiliza congelado para todo.
FROZEN_SCALER = fit_sequence_scaler(np.concatenate([X_islr, Xm_islr]))


def train_model(model, X_tr, y_tr, epochs, lr, batch_size=32, seed=SEED):
    """Entrena sin validacion: epocas fijas, escalador congelado."""
    seed_everything(seed)
    loader = make_loader(apply_sequence_scaler(X_tr, FROZEN_SCALER), y_tr, train=True, batch_size=batch_size)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.03)
    for ep in range(1, epochs + 1):
        model.train(); tot, n = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item()) * len(yb); n += len(yb)
        sched.step()
        print(f'  epoca {ep}/{epochs} loss={tot / max(n, 1):.4f}')
    return model


def proba(model, X, Xm):
    p = predict_tcn(model, apply_sequence_scaler(X, FROZEN_SCALER))
    return (p + predict_tcn(model, apply_sequence_scaler(Xm, FROZEN_SCALER))) / 2.0


## 4. Preentrenamiento en ISLR (250 senas)

In [ ]:
n_cls = len(sign_to_id)
pretrained = LSC5_TCN(num_classes=n_cls)
pretrained = train_model(pretrained, np.concatenate([X_islr, Xm_islr]), np.concatenate([y_islr, y_islr]),
                         PRETRAIN_EPOCHS, PRETRAIN_LR, batch_size=PRETRAIN_BATCH)
torch.save(pretrained.state_dict(), OUT_DIR / 'tcn_pretrained_islr.pt')
with open(OUT_DIR / 'islr_sign_to_id.json', 'w', encoding='utf-8') as f:
    json.dump(sign_to_id, f, ensure_ascii=False)
print('Preentrenamiento listo. Pesos guardados en', OUT_DIR)


## 5. LOSO en LSC5: desde cero vs preentrenado (misma configuracion y semillas)

Cada fold reserva un participante completo. Se entrena con los otros dos (con espejo) y se evalua con promedio de predicciones original/espejo. Las dos variantes usan el mismo escalador congelado, las mismas epocas y la misma tasa de aprendizaje; solo cambia la inicializacion del modelo.

In [ ]:
def init_from_pretrained(state):
    m = LSC5_TCN(num_classes=5)
    own = m.state_dict()
    keep = {k: v for k, v in state.items() if k in own and v.shape == own[k].shape}  # descarta la cabeza de 250 clases
    own.update(keep); m.load_state_dict(own)
    return m, len(keep), len(own)


state0 = torch.load(OUT_DIR / 'tcn_pretrained_islr.pt', map_location='cpu')
rows = []
for seed in SEEDS:
    for k, held in enumerate(sorted(set(signers))):
        tr = np.where(signers != held)[0]; va = np.where(signers == held)[0]
        Xt = np.concatenate([X_l[tr], Xm_l[tr]]); yt = np.concatenate([y_l[tr], y_l[tr]])
        for name in ('scratch', 'pretrained'):
            if name == 'scratch':
                seed_everything(seed + k); model = LSC5_TCN(num_classes=5)
            else:
                model, nk, nt = init_from_pretrained(state0)
            print(f'seed={seed} reservado={held} {name}' + (f' (pesos cargados {nk}/{nt})' if name == 'pretrained' else ''))
            model = train_model(model, Xt, yt, FT_EPOCHS, FT_LR, seed=seed + k)
            pred = proba(model, X_l[va], Xm_l[va]).argmax(1)
            m = calculate_metrics(y_l[va], pred)
            rows.append({'seed': seed, 'heldout': held, 'init': name, **m})
            print(f"   accuracy={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f}")

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / 'loso_scratch_vs_pretrained.csv', index=False)
summary = res.groupby(['init', 'heldout']).accuracy.mean().unstack(0).round(4)
summary.loc['MEDIA'] = summary.mean()
display(summary)
print('Accuracy media por inicializacion:')
print(res.groupby('init').accuracy.agg(['mean', 'std']).round(4))


## 6. Como leer el resultado

- Si `pretrained` supera a `scratch` en la media **y** en el peor participante, de forma consistente entre semillas, el preentrenamiento aporta.
- Si la diferencia es pequena o inconsistente, no vale la pena el riesgo de usar datos externos: se descarta.
- Cualquier uso en la entrega requiere declarar ISLR (fuente, licencia) en el reporte y la aprobacion del comite.